In [0]:
#Import libraries
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType, IntegerType
from pyspark.sql.functions import trim,col

In [0]:
# Read data bronze table and load to dataframe
df = spark.table("baraa_projectwork.bronze.erp_cust_az12")


In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

In [0]:
df = df.withColumn(
    "cid",
    F.when(col("cid").startswith("NAS"),
           F.substring(col("cid"), 4, F.length(col("cid"))))
     .otherwise(col("cid"))
)

In [0]:
df = df.withColumn(
    "bdate",
    F.when(col("bdate") > F.current_date(), None)
     .otherwise(col("bdate"))
)

In [0]:
df = df.withColumn(
    "gen",
    F.when(F.upper(col("gen")).isin("F", "FEMALE"), "Female")
     .when(F.upper(col("gen")).isin("M", "MALE"), "Male")
     .otherwise("n/a")
)
     

In [0]:
RENAME_MAP = {
    "CID": "customer_number",
    "BDATE": "birth_date",
    "GEN": "gender"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

### Writing to Silver table


In [0]:
df.limit(10).display()

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("baraa_projectwork.silver.erp_customers")

In [0]:

%sql
SELECT * FROM baraa_projectwork.silver.erp_customers LIMIT 10